# Create a generative AI chat app

## 1. CLI Azure

Setup Your Development Environment

In [ ]:
az login

Create a `.env` file based on `.env.example`:

In [ ]:
AZURE_OPENAI_ENDPOINT=https://your-openai-resource.openai.azure.com/
AZURE_OPENAI_API_KEY=your-openai-api-key
AZURE_OPENAI_MODEL_DEPLOYMENT=gpt41-deployment
AZURE_SUBSCRIPTION_ID=your-subscription-id
AZURE_RESOURCE_GROUP=rg-chat-app-cli
AZURE_LOCATION=eastus2
AZURE_AI_HUB_NAME=chat-app-hub
AZURE_AI_PROJECT_NAME=chat-app-project

Create a resource group:

In [ ]:
az group create --name rg-chat-app-cli --location eastus2

Create an Azure AI Foundry hub:

In [ ]:
az ml workspace create --name chat-app-hub --resource-group rg-chat-app-cli --location eastus2 --display-name "Chat App Hub Display"

Create an Azure AI Services resource for OpenAI:

In [ ]:
az cognitiveservices account create --name chat-app-hub-aiservices --resource-group rg-chat-app-cli --location eastus2 --kind OpenAI --sku S0

Create a hub-based project:

[View the Azure AI Foundry documentation for more information.](https://learn.microsoft.com/en-us/azure/ai-foundry/how-to/create-projects?pivots=hub-project&tabs=azurecli)

In [ ]:
az ml workspace create --name chat-app-project --resource-group rg-chat-app-cli --location eastus2 --hub-id "/subscriptions/562be4bd-42dc-4fb6-8e07-e3fd5400b9f7/resourceGroups/rg-chat-app-cli/providers/Microsoft.MachineLearningServices/workspaces/chat-app-hub" --kind project --display-name "Chat App Project Display"

List Models Avalible

In [ ]:
az cognitiveservices account list-models -n chat-app-hub-aiservices -g rg-chat-app-cli | jq '.[] | { name: .name, format: .format, version: .version, sku: .skus[0].name, capacity: .skus[0].capacity.default }'

output `json`

In [ ]:
...
{
  "name": "gpt-4.1",
  "format": "OpenAI",
  "version": "2025-04-14",
  "sku": "GlobalStandard",
  "capacity": 10
}
...

Create an online endpoint:

In [ ]:
az ml online-endpoint create \
  --name chat-app-hub-endpoint \
  --resource-group rg-chat-app-cli \
  --workspace-name chat-app-hub


Deploy the `gpt-4.1` model:

In [ ]:
az ml online-deployment create \
  --file gpt41-deployment.yml \
  --resource-group rg-chat-app-cli \
  --workspace-name chat-app-hub

## 2. SDk Python

## El deployment de modelos se realiza con el CLI Azure. 

In [ ]:
az cognitiveservices account deployment create \
    -g rg-gpt41-sdk \
    -n opengpt41-foundry-rs \
    --deployment-name gpt41-deployment \
    --model-name gpt-4.1 \
    --model-version "2025-04-14" \
    --model-format OpenAI \
    --sku-capacity 1 \
    --sku-name GlobalStandard

## Obtener endpoint

In [9]:
account_name = "opengpt41-foundry-rs"
account = client.accounts.get(resource_group, account_name)
endpoint = account.properties.endpoints["Azure AI Model Inference API"]

In [10]:
print(endpoint)

https://opengpt41-foundry-rs.services.ai.azure.com/


In [ ]:
# Obtener el endpoint
account = client.accounts.get(resource_group, account_name)
endpoint = account.properties.endpoints["Azure AI Model Inference API"]

# Construir la URL completa
full_endpoint = f"{endpoint}openai/deployments/{deployment_name}/chat/completions?api-version={api_version}"

print(f"Endpoint completo: {full_endpoint}")

# Opcional: Obtener la clave
keys = client.accounts.list_keys(resource_group, account_name)
print(f"Clave de API: {keys.key1 or keys.key2}")